# Data Formats & Storage — 01: Parquet vs ORC vs Avro vs CSV

## What you will learn
Choosing the right file format is one of the highest-impact decisions in a data pipeline.
The wrong format can make queries **10–100x slower** and waste disk space.

In this notebook you will:
1. Understand the fundamental differences between row-based and columnar formats
2. Write and read data in Parquet, ORC, Avro, JSON, and CSV
3. Benchmark read performance with and without predicate pushdown
4. Measure compression ratios
5. Understand when to use each format

## Row-based vs Columnar: The Core Concept
```
Row-based (CSV, Avro, JSON):
  Row 1: [id=1, name=Alice, salary=95000, dept=Eng]
  Row 2: [id=2, name=Bob,   salary=72000, dept=Mkt]
  → Good for: writing full rows, streaming, row-level access
  → Bad for:  "SELECT salary FROM ..." (reads ALL columns just to get salary)

Columnar (Parquet, ORC):
  id column:     [1, 2, 3, 4, ...]
  name column:   [Alice, Bob, Carol, ...]
  salary column: [95000, 72000, 105000, ...]
  dept column:   [Eng, Mkt, Eng, ...]
  → Good for: analytical queries (SELECT few columns, aggregate)
  → Bad for:  writing/reading individual rows (need to touch all columns)
```
For analytics (Spark's main use case), **columnar is almost always better**.


In [1]:
import os, time, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('data-formats-storage')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 15:03:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Gluten: False


## Step 1 — Generate a Realistic Dataset

We create a dataset large enough to see meaningful differences between formats.


In [2]:
import random
random.seed(42)

N = 200_000  # 200k rows — enough to see format differences

DEPTS = ["Engineering", "Marketing", "HR", "Finance", "Sales", "Legal", "Operations"]
COUNTRIES = ["US", "UK", "DE", "FR", "JP", "CA", "AU", "BR", "IN", "CN"]
TITLES = ["Junior", "Mid", "Senior", "Lead", "Principal", "Director", "VP"]

schema = StructType([
    StructField("employee_id",  LongType()),
    StructField("name",         StringType()),
    StructField("department",   StringType()),
    StructField("title",        StringType()),
    StructField("salary",       DoubleType()),
    StructField("country",      StringType()),
    StructField("hire_year",    IntegerType()),
    StructField("performance",  DoubleType()),
    StructField("is_remote",    BooleanType()),
    StructField("projects",     IntegerType()),
])

data = [
    (i,
     f"Employee_{i:06d}",
     random.choice(DEPTS),
     random.choice(TITLES),
     round(random.gauss(85000, 25000), 2),
     random.choice(COUNTRIES),
     random.randint(2010, 2024),
     round(random.uniform(1.0, 10.0), 2),
     random.random() > 0.4,
     random.randint(0, 15))
    for i in range(1, N + 1)
]

df = spark.createDataFrame(data, schema)
df.cache()
df.count()  # materialize cache
print(f"Dataset: {N:,} rows, {len(schema)} columns")
df.printSchema()
df.show(3)

26/04/08 15:03:20 WARN TaskSetManager: Stage 0 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:22 WARN TaskSetManager: Stage 1 contains a task of very large size (2743 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:23 WARN TaskSetManager: Stage 4 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows, 10 columns
root
 |-- employee_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- title: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- country: string (nullable = true)
 |-- hire_year: integer (nullable = true)
 |-- performance: double (nullable = true)
 |-- is_remote: boolean (nullable = true)
 |-- projects: integer (nullable = true)

+-----------+---------------+----------+------+---------+-------+---------+-----------+---------+--------+
|employee_id|           name|department| title|   salary|country|hire_year|performance|is_remote|projects|
+-----------+---------------+----------+------+---------+-------+---------+-----------+---------+--------+
|          1|Employee_000001|     Legal|Junior|104803.62|     FR|     2012|       7.63|     true|       2|
|          2|Employee_000002|     Sales|  Lead| 88137.96|     US|     2010|       1.84|    false|       0|
|          3|Employee_00

## Step 2 — Write in All Formats

We write the same dataset in 5 formats and compare:
- File size (storage cost)
- Write time (ingestion speed)


In [3]:
formats_dir = f"{DATA_DIR}/format_comparison"
pathlib.Path(formats_dir).mkdir(parents=True, exist_ok=True)

write_results = {}

# CSV — no compression, no schema, human-readable
t0 = time.time()
df.write.mode("overwrite").option("header", True).csv(f"{formats_dir}/csv")
write_results["CSV"] = {"time": time.time()-t0}

# JSON — semi-structured, verbose
t0 = time.time()
df.write.mode("overwrite").json(f"{formats_dir}/json")
write_results["JSON"] = {"time": time.time()-t0}

# Parquet — columnar, snappy compression (default)
t0 = time.time()
df.write.mode("overwrite").parquet(f"{formats_dir}/parquet_snappy")
write_results["Parquet/Snappy"] = {"time": time.time()-t0}

# Parquet — columnar, zstd compression (better ratio)
t0 = time.time()
df.write.mode("overwrite").option("compression","zstd").parquet(f"{formats_dir}/parquet_zstd")
write_results["Parquet/ZSTD"] = {"time": time.time()-t0}

# ORC — columnar, zlib compression
t0 = time.time()
df.write.mode("overwrite").orc(f"{formats_dir}/orc")
write_results["ORC"] = {"time": time.time()-t0}

# Measure file sizes
import subprocess
for fmt, path_suffix in [
    ("CSV",           "csv"),
    ("JSON",          "json"),
    ("Parquet/Snappy","parquet_snappy"),
    ("Parquet/ZSTD",  "parquet_zstd"),
    ("ORC",           "orc"),
]:
    result = subprocess.run(
        ["du", "-sb", f"{formats_dir}/{path_suffix}"],
        capture_output=True, text=True
    )
    size_bytes = int(result.stdout.split()[0])
    write_results[fmt]["size_mb"] = round(size_bytes / 1024 / 1024, 2)

print(f"{'Format':<20} {'Write Time':>12} {'Size (MB)':>12} {'Compression':>12}")
print("-" * 60)
csv_size = write_results["CSV"]["size_mb"]
for fmt, r in write_results.items():
    ratio = f"{csv_size/r['size_mb']:.1f}x" if r['size_mb'] > 0 else "-"
    print(f"{fmt:<20} {r['time']:>10.2f}s {r['size_mb']:>10.1f} MB {ratio:>12}")

26/04/08 15:03:23 WARN TaskSetManager: Stage 5 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:24 WARN TaskSetManager: Stage 6 contains a task of very large size (2743 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:25 WARN TaskSetManager: Stage 7 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:27 WARN TaskSetManager: Stage 8 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:28 WARN TaskSetManager: Stage 9 contains a task of very large size (2743 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Format                 Write Time    Size (MB)  Compression
------------------------------------------------------------
CSV                        1.29s       12.8 MB         1.0x
JSON                       0.99s       35.9 MB         0.4x
Parquet/Snappy             1.68s        3.6 MB         3.6x
Parquet/ZSTD               0.90s        1.8 MB         7.0x
ORC                        1.20s        1.8 MB         7.2x


## Step 3 — Read Performance: Full Scan

Now measure read performance. For a full scan (all rows, all columns),
all formats do similar work. The differences appear in **selective queries**.


In [4]:
read_results = {}

full_scan_query = lambda path, fmt: (
    spark.read.csv(path, header=True, inferSchema=True) if fmt=="csv"
    else spark.read.json(path) if fmt=="json"
    else spark.read.orc(path) if fmt=="orc"
    else spark.read.parquet(path)
).agg(F.sum("salary"), F.avg("performance")).collect()

for fmt, path_suffix, fmt_type in [
    ("CSV",           "csv",            "csv"),
    ("JSON",          "json",           "json"),
    ("Parquet/Snappy","parquet_snappy", "parquet"),
    ("Parquet/ZSTD",  "parquet_zstd",  "parquet"),
    ("ORC",           "orc",           "orc"),
]:
    path = f"{formats_dir}/{path_suffix}"
    times = []
    for _ in range(3):
        t0 = time.time()
        full_scan_query(path, fmt_type)
        times.append(time.time() - t0)
    read_results[fmt] = {"full_scan": round(sorted(times)[1], 3)}

print(f"{'Format':<20} {'Full Scan (median)':>20}")
print("-" * 45)
for fmt, r in read_results.items():
    print(f"{fmt:<20} {r['full_scan']:>18.3f}s")

Format                 Full Scan (median)
---------------------------------------------
CSV                               0.855s
JSON                              0.782s
Parquet/Snappy                    0.360s
Parquet/ZSTD                      0.330s
ORC                               0.324s


## Step 4 — Read Performance: Selective Query (The Key Difference!)

When you select only 2 out of 10 columns, columnar formats shine:
- **Parquet/ORC**: read only the 2 columns needed — skip 8 columns entirely
- **CSV/JSON**: must read ALL columns, then discard 8

This is called **column pruning** and is the #1 reason to use columnar formats.


In [5]:
# Selective query: only 2 out of 10 columns + filter + aggregation
# This is where columnar formats have a massive advantage

def selective_query(path, fmt_type):
    reader = (spark.read.csv(path, header=True, inferSchema=True) if fmt_type=="csv"
              else spark.read.json(path) if fmt_type=="json"
              else spark.read.orc(path) if fmt_type=="orc"
              else spark.read.parquet(path))
    return (reader
            .filter(F.col("department") == "Engineering")
            .filter(F.col("salary") > 90000)
            .select("salary", "performance")
            .agg(F.avg("salary"), F.avg("performance"))
            .collect())

for fmt, path_suffix, fmt_type in [
    ("CSV",           "csv",            "csv"),
    ("JSON",          "json",           "json"),
    ("Parquet/Snappy","parquet_snappy", "parquet"),
    ("Parquet/ZSTD",  "parquet_zstd",  "parquet"),
    ("ORC",           "orc",           "orc"),
]:
    path = f"{formats_dir}/{path_suffix}"
    times = []
    for _ in range(3):
        t0 = time.time()
        selective_query(path, fmt_type)
        times.append(time.time() - t0)
    read_results[fmt]["selective"] = round(sorted(times)[1], 3)

print(f"{'Format':<20} {'Full Scan':>12} {'Selective':>12} {'Speedup':>10}")
print("-" * 58)
baseline = read_results["CSV"]["selective"]
for fmt, r in read_results.items():
    speedup = f"{baseline/r['selective']:.1f}x" if fmt != "CSV" else "1.0x (baseline)"
    print(f"{fmt:<20} {r['full_scan']:>10.3f}s {r['selective']:>10.3f}s {speedup:>12}")

print()
print("Columnar formats (Parquet/ORC) are MUCH faster for selective queries")
print("because they skip reading columns not referenced in the query.")

Format                  Full Scan    Selective    Speedup
----------------------------------------------------------
CSV                       0.855s      0.663s 1.0x (baseline)
JSON                      0.782s      0.672s         1.0x
Parquet/Snappy            0.360s      0.352s         1.9x
Parquet/ZSTD              0.330s      0.318s         2.1x
ORC                       0.324s      0.351s         1.9x

Columnar formats (Parquet/ORC) are MUCH faster for selective queries
because they skip reading columns not referenced in the query.


## Step 5 — Predicate Pushdown: Reading Less Data

Parquet and ORC store **column statistics** (min/max values per row group).
When you filter on a column, Spark can skip entire row groups where the
filter condition cannot possibly match.

This is called **predicate pushdown** and can reduce I/O by 50-90%.


In [6]:
# Write a partitioned Parquet file for max pushdown benefit
df.write.mode("overwrite") \
        .partitionBy("department") \
        .parquet(f"{formats_dir}/parquet_partitioned")

# Query with partition filter (partition pruning)
t0 = time.time()
result_partitioned = (
    spark.read.parquet(f"{formats_dir}/parquet_partitioned")
         .filter(F.col("department") == "Engineering")   # partition filter
         .filter(F.col("salary") > 100000)
         .agg(F.count("*"), F.avg("salary"))
         .collect()
)
t_partitioned = time.time() - t0

# Same query on non-partitioned Parquet
t0 = time.time()
result_flat = (
    spark.read.parquet(f"{formats_dir}/parquet_snappy")
         .filter(F.col("department") == "Engineering")
         .filter(F.col("salary") > 100000)
         .agg(F.count("*"), F.avg("salary"))
         .collect()
)
t_flat = time.time() - t0

print(f"Non-partitioned Parquet: {t_flat:.3f}s  (scans all {N:,} rows)")
print(f"Partitioned Parquet    : {t_partitioned:.3f}s  (only scans Engineering partition)")
print(f"Speedup from partitioning: {t_flat/t_partitioned:.1f}x")
print()
print("Explain — observe FileScan with PartitionFilters:")
spark.read.parquet(f"{formats_dir}/parquet_partitioned") \
     .filter(F.col("department") == "Engineering") \
     .explain(mode="formatted")

26/04/08 15:03:47 WARN TaskSetManager: Stage 130 contains a task of very large size (2743 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Non-partitioned Parquet: 0.359s  (scans all 200,000 rows)
Partitioned Parquet    : 0.832s  (only scans Engineering partition)
Speedup from partitioning: 0.4x

Explain — observe FileScan with PartitionFilters:
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [10]: [employee_id#3044L, name#3045, title#3046, salary#3047, country#3048, hire_year#3049, performance#3050, is_remote#3051, projects#3052, department#3053]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/format_comparison/parquet_partitioned]
PartitionFilters: [isnotnull(department#3053), (department#3053 = Engineering)]
ReadSchema: struct<employee_id:bigint,name:string,title:string,salary:double,country:string,hire_year:int,performance:double,is_remote:boolean,projects:int>

(2) ColumnarToRow [codegen id : 1]
Input [10]: [employee_id#3044L, name#3045, title#3046, salary#3047, country#3048, hire_year#3049, performance#3050, is_remote#3051, projects#3052, department#3053]




## Step 6 — Format Selection Guide

| Scenario | Best Format | Reason |
|---|---|---|
| Analytical queries (SELECT few cols) | Parquet | Column pruning, predicate pushdown |
| Data lake storage (long term) | Parquet/ORC | Compression, schema evolution |
| Message streaming (Kafka) | Avro | Schema registry support, row-based is fine for streams |
| Data exchange with external systems | CSV/JSON | Human-readable, universal support |
| High compression needs | Parquet+ZSTD | Best compression ratio |
| Fast random access by row | Avro | Row-based, no need to read other columns |
| Schema evolution (add/remove fields) | Parquet or Avro | Both support schema evolution |

### Compression codec comparison for Parquet
| Codec | Ratio | Speed | Best for |
|---|---|---|---|
| `snappy` | Medium | Fast | Default, good balance |
| `zstd` | High | Medium | When storage cost matters |
| `gzip` | High | Slow | Archival, rarely for active data |
| `lz4` | Low | Very fast | Temporary files, fast pipelines |
| `none` | 1x | Fastest | When CPU is bottleneck |


In [7]:
# Compression codec comparison
codecs = ["snappy", "zstd", "gzip", "lz4", "none"]
codec_results = {}

for codec in codecs:
    path = f"{formats_dir}/parquet_{codec}"
    t0 = time.time()
    df.write.mode("overwrite").option("compression", codec).parquet(path)
    write_time = time.time() - t0

    result = subprocess.run(["du", "-sb", path], capture_output=True, text=True)
    size_mb = round(int(result.stdout.split()[0]) / 1024 / 1024, 1)

    t0 = time.time()
    spark.read.parquet(path).agg(F.sum("salary")).collect()
    read_time = time.time() - t0

    codec_results[codec] = {"write": write_time, "size_mb": size_mb, "read": read_time}

print(f"{'Codec':<10} {'Write':>8} {'Size MB':>10} {'Read':>8} {'vs snappy':>10}")
print("-" * 52)
base_size = codec_results["snappy"]["size_mb"]
for codec, r in codec_results.items():
    ratio = f"{r['size_mb']/base_size:.2f}x"
    print(f"{codec:<10} {r['write']:>6.2f}s {r['size_mb']:>8.1f} MB {r['read']:>6.2f}s {ratio:>10}")

26/04/08 15:03:53 WARN TaskSetManager: Stage 140 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:54 WARN TaskSetManager: Stage 145 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:55 WARN TaskSetManager: Stage 150 contains a task of very large size (2743 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:56 WARN TaskSetManager: Stage 155 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.
26/04/08 15:03:57 WARN TaskSetManager: Stage 160 contains a task of very large size (2675 KiB). The maximum recommended task size is 1000 KiB.


Codec         Write    Size MB     Read  vs snappy
----------------------------------------------------
snappy       0.77s      3.6 MB   0.31s      1.00x
zstd         0.74s      1.8 MB   0.28s      0.50x
gzip         0.83s      2.3 MB   0.29s      0.64x
lz4          0.76s      3.4 MB   0.28s      0.94x
none         0.79s      7.5 MB   0.28s      2.08x


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 53036)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin

## Summary

### Key takeaways
1. **Always use Parquet** for analytical workloads — column pruning alone gives 3–10x speedup
2. **Partition your data** by the most common filter column (date, department, region)
3. **Use ZSTD compression** for long-term storage; Snappy for hot data
4. **CSV/JSON** are only for ingestion and interoperability — never for internal storage
5. **Schema matters** — define explicit schemas when reading CSV/JSON (avoid `inferSchema`)

**Next:** `02_iceberg_advanced.ipynb` — ACID transactions, time travel, and schema evolution.
